# Deep Learning HW2 - Siamese Network Training

## Step 1: Setup Environment
Mount your Google Drive where you uploaded the `DL_HW2.zip` file.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 2: Unzip Project
This unzips your project directly into the Colab runtime. This is **faster** for training than reading directly from Drive.

In [ ]:
import os
# EXAMPLE PATH - CHANGE THIS to where your zip is
zip_path = '/content/drive/MyDrive/DL_HW2.zip' 

# Unzip to /content/project (local runtime disk)
!unzip -q "$zip_path" -d /content/project

# Navigate to the unzipped folder
project_path = '/content/project/DL_HW2'
if os.path.exists(project_path):
    os.chdir(project_path)
else:
    os.chdir('/content/project')

print(f"Current Working Directory: {os.getcwd()}")

## Step 3: Install Dependencies

In [ ]:
!pip install -r requirements.txt

In [ ]:
# Verify GPU is available
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
else:
    print("❌ No GPU found. Go to Runtime -> Change runtime type -> T4 GPU")

## Step 4: Dataset Analysis
Generate statistics for the report.

In [ ]:
!python src/utils/analyze_dataset.py

## Step 4b: Name this run
Every run writes to its own folder on Drive. **Change `RUN_NAME` for every new experiment.** The cell refuses to reuse a folder that already holds finished results, so a later run cannot overwrite an earlier one.

In [ ]:
import os
RUN_NAME = 'run1_cosine_head'   # CHANGE for every experiment
RUN_DIR = f'/content/drive/MyDrive/DL_HW2_runs/{RUN_NAME}'
if os.path.exists(f'{RUN_DIR}/eval/results.json'):
    raise RuntimeError(f'{RUN_DIR} already holds finished results; pick a new RUN_NAME')
os.makedirs(RUN_DIR, exist_ok=True)
print('Run folder:', RUN_DIR)

## Step 5: Start TensorBoard
Monitor training progress in real-time.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{RUN_DIR}/logs"

## Step 6a: Run the test suite
Checks the sigmoid fix, metrics, fixed-pair determinism, and that the committed `data/splits/` files match what the split code regenerates. Do not train if this fails.

In [ ]:
!python -m pytest -q

## Step 6a+: Smoke test (a few minutes)
One epoch capped at 5 batches, then evaluate that checkpoint. Exercises training with the cosine head, checkpoint saving and evaluate.py loading, before spending GPU time on the full run. Writes to `/content/smoke` (local, thrown away).

In [ ]:
!rm -rf /content/smoke
!python train.py --epochs 1 --max-batches 5 --num-workers 2 --checkpoint-dir /content/smoke/checkpoints --log-dir /content/smoke/logs
!python evaluate.py --checkpoint /content/smoke/checkpoints/best_model.pth --out /content/smoke/eval/results.json --num-workers 2

## Step 6b: Run Training
Trains on `data/splits/train_people.txt` and keeps the checkpoint with the best val AUC (val pairs only; test is never read). Checkpoints and logs go **straight to Drive** under `RUN_DIR`, so a runtime disconnect does not lose them.

In [ ]:
!python train.py --num-workers 2 --checkpoint-dir "{RUN_DIR}/checkpoints" --log-dir "{RUN_DIR}/logs"

## Step 6c: Evaluate (test touched once)
Tunes the threshold on val, then reports test accuracy, AUC and 95% CI. Results are written to `RUN_DIR/eval/`.

In [ ]:
!python evaluate.py --checkpoint "{RUN_DIR}/checkpoints/best_model.pth" --out "{RUN_DIR}/eval/results.json" --num-workers 2
!cat "{RUN_DIR}/eval/results.json"

## Step 7: Save Results
Checkpoints, logs and eval results are already on Drive in `RUN_DIR`. This only adds the dataset plot. Nothing outside `RUN_DIR` is written, so earlier runs (including the baseline) are never overwritten.

In [ ]:
!cp dataset_distribution.png "{RUN_DIR}/"
!ls -R "{RUN_DIR}" | head -40